# This Notebook is to run STREME/MEME, TOMTOM and FIMO

## Set UP

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
import os
from datetime import datetime
import urllib.request

In [ ]:
PROJECT = Path("/s/project/ml4rg_students/2026/project15")

FASTA_DIR = PROJECT / "working" / "sequence_datasets_fastas"
RESULT_DIR = PROJECT / "working" / "streme_results"
LOG_DIR = PROJECT / "working" / "logs" / "streme_notebook"

MEME_ENV = Path("/opt/modules/i12g/anaconda/envs/meme_env")
MEME_BIN = MEME_ENV / "bin"

STREME = MEME_BIN / "meme"
TOMTOM = MEME_BIN / "tomtom"
FIMO = MEME_BIN / "fimo"

JASPAR_DIR = PROJECT / "working" / "jaspar"
JASPAR_DIR.mkdir(parents=True, exist_ok=True)

JASPAR_FUNGI = JASPAR_DIR / "JASPAR2026_CORE_fungi_non-redundant_pfms_meme.txt"

url = "https://jaspar.elixir.no/download/data/2026/CORE/JASPAR2026_CORE_fungi_non-redundant_pfms_meme.txt"

if not JASPAR_FUNGI.exists() or JASPAR_FUNGI.stat().st_size == 0:
    urllib.request.urlretrieve(url, JASPAR_FUNGI)

print(JASPAR_FUNGI)
print("exists:", JASPAR_FUNGI.exists())
print("size:", JASPAR_FUNGI.stat().st_size)

for d in [
    RESULT_DIR / "streme",
    RESULT_DIR / "tomtom",
    RESULT_DIR / "fimo_jaspar",
    RESULT_DIR / "fimo_streme",
    LOG_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("FASTA_DIR:", FASTA_DIR)
print("RESULT_DIR:", RESULT_DIR)
print("JASPAR:", JASPAR)

## Preperation

In [ ]:
for tool in [STREME, TOMTOM, FIMO, JASPAR]:
    print(tool, "exists:", tool.exists())

In [ ]:
fastas = sorted(list(FASTA_DIR.glob("*.fa")) + list(FASTA_DIR.glob("*.fasta")))

print("Number of FASTA files:", len(fastas))
for f in fastas[:10]:
    print(f.name)

In [ ]:
def fasta_name(fasta_path: Path) -> str:
    name = fasta_path.name
    if name.endswith(".fasta"):
        name = name[:-6]
    elif name.endswith(".fa"):
        name = name[:-3]
    return name


def paths_for_fasta(fasta_path: Path):
    name = fasta_name(fasta_path)
    
    streme_out = RESULT_DIR / "streme" / name
    tomtom_out = RESULT_DIR / "tomtom" / name
    fimo_jaspar_out = RESULT_DIR / "fimo_jaspar" / name
    fimo_streme_out = RESULT_DIR / "fimo_streme" / name
    
    return {
        "name": name,
        "streme_out": streme_out,
        "tomtom_out": tomtom_out,
        "fimo_jaspar_out": fimo_jaspar_out,
        "fimo_streme_out": fimo_streme_out,
        "streme_txt": streme_out / "streme.txt",
        "tomtom_tsv": tomtom_out / "tomtom.tsv",
        "fimo_jaspar_tsv": fimo_jaspar_out / "fimo.tsv",
        "fimo_streme_tsv": fimo_streme_out / "fimo.tsv",
    }


def exists_nonempty(path: Path) -> bool:
    return path.exists() and path.stat().st_size > 0


def run_command(cmd, log_prefix: Path):
    """
    Runs a command and writes stdout/stderr to log files.
    Raises an error if the command fails.
    """
    log_prefix.parent.mkdir(parents=True, exist_ok=True)
    
    stdout_file = log_prefix.with_suffix(".out")
    stderr_file = log_prefix.with_suffix(".err")
    
    print("Running:")
    print(" ".join(map(str, cmd)))
    print("stdout:", stdout_file)
    print("stderr:", stderr_file)
    
    with open(stdout_file, "w") as out, open(stderr_file, "w") as err:
        result = subprocess.run(
            list(map(str, cmd)),
            stdout=out,
            stderr=err,
            text=True,
        )
    
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}. Check {stderr_file}"
        )

### Check status


In [ ]:
rows = []

for fasta in fastas:
    p = paths_for_fasta(fasta)
    rows.append({
        "name": p["name"],
        "fasta": str(fasta),
        "streme": exists_nonempty(p["streme_txt"]),
        "tomtom": exists_nonempty(p["tomtom_tsv"]),
        "fimo_jaspar": exists_nonempty(p["fimo_jaspar_tsv"]),
        "fimo_streme": exists_nonempty(p["fimo_streme_tsv"]),
    })

status = pd.DataFrame(rows)
status

In [ ]:
status[["streme", "tomtom", "fimo_jaspar", "fimo_streme"]].sum()

## Running Pipeline

In [ ]:
def run_pipeline_for_fasta(
    fasta: Path,
    force: bool = False,
    streme_time: int = 1800,
    minw: int = 6,
    maxw: int = 30,
    nmotifs: int = 20,
    fimo_thresh: str = "1e-4",
):
    p = paths_for_fasta(fasta)
    name = p["name"]
    
    print("=" * 80)
    print(f"Processing: {name}")
    print(f"FASTA: {fasta}")
    print("=" * 80)
    
    for key in ["streme_out", "tomtom_out", "fimo_jaspar_out", "fimo_streme_out"]:
        p[key].mkdir(parents=True, exist_ok=True)
    
    # -------------------------
    # STREME
    # -------------------------
    if exists_nonempty(p["streme_txt"]) and not force:
        print("STREME exists, skipping.")
    else:
        print("Running STREME.")
        
        cmd = [
            STREME,
            "--dna",
            "--p", fasta,
            "--oc", p["streme_out"],
            "--minw", minw,
            "--maxw", maxw,
            "--nmotifs", nmotifs,
            "--time", streme_time,
            "--verbosity", 1,
        ]
        
        run_command(cmd, LOG_DIR / f"{name}.streme")
    
    if not exists_nonempty(p["streme_txt"]):
        print("STREME result missing. Skipping TOMTOM and FIMO_STREME.")
        return
    
    # -------------------------
    # TOMTOM
    # -------------------------
    if exists_nonempty(p["tomtom_tsv"]) and not force:
        print("TOMTOM exists, skipping.")
    else:
        print("Running TOMTOM.")
        
        cmd = [
            TOMTOM,
            "-oc", p["tomtom_out"],
            "-verbosity", 1,
            "-min-overlap", 5,
            "-dist", "pearson",
            "-evalue",
            "-thresh", 10,
            p["streme_txt"],
            JASPAR,
        ]
        
        run_command(cmd, LOG_DIR / f"{name}.tomtom")
    
    # -------------------------
    # FIMO with JASPAR motifs
    # -------------------------
    if exists_nonempty(p["fimo_jaspar_tsv"]) and not force:
        print("FIMO JASPAR exists, skipping.")
    else:
        print("Running FIMO with JASPAR.")
        
        cmd = [
            FIMO,
            "--oc", p["fimo_jaspar_out"],
            "--thresh", fimo_thresh,
            "--max-stored-scores", 1000000,
            JASPAR,
            fasta,
        ]
        
        run_command(cmd, LOG_DIR / f"{name}.fimo_jaspar")
    
    # -------------------------
    # FIMO with STREME motifs
    # -------------------------
    if exists_nonempty(p["fimo_streme_tsv"]) and not force:
        print("FIMO STREME exists, skipping.")
    else:
        print("Running FIMO with STREME motifs.")
        
        cmd = [
            FIMO,
            "--oc", p["fimo_streme_out"],
            "--thresh", fimo_thresh,
            "--max-stored-scores", 1000000,
            p["streme_txt"],
            fasta,
        ]
        
        run_command(cmd, LOG_DIR / f"{name}.fimo_streme")
    
    print(f"Finished: {name}")

### To test just for one FASTA

In [ ]:
test_fasta = fastas[0]
test_fasta

In [ ]:
run_pipeline_for_fasta(
    test_fasta,
    force=False,
    streme_time=1800,
    minw=6,
    maxw=30,
    nmotifs=20,
    fimo_thresh="1e-4",
)

### For all FASTAS

In [ ]:
for i, fasta in enumerate(fastas, start=1):
    print(f"\n### {i}/{len(fastas)} ###")
    
    try:
        run_pipeline_for_fasta(
            fasta,
            force=False,
            streme_time=1800,
            minw=6,
            maxw=20,
            nmotifs=10,
            fimo_thresh="1e-4",
        )
    except Exception as e:
        print(f"ERROR for {fasta.name}: {e}")
        print("Continuing with next FASTA.")

### Update status

In [ ]:
rows = []

for fasta in fastas:
    p = paths_for_fasta(fasta)
    rows.append({
        "name": p["name"],
        "streme": exists_nonempty(p["streme_txt"]),
        "tomtom": exists_nonempty(p["tomtom_tsv"]),
        "fimo_jaspar": exists_nonempty(p["fimo_jaspar_tsv"]),
        "fimo_streme": exists_nonempty(p["fimo_streme_tsv"]),
    })

status = pd.DataFrame(rows)
status

In [ ]:
status[["streme", "tomtom", "fimo_jaspar", "fimo_streme"]].sum()

In [ ]:
status[~(status["streme"] & status["tomtom"] & status["fimo_jaspar"] & status["fimo_streme"])]

## Import Results

### TOMTOM

In [ ]:
tomtom_tables = []

for fasta in fastas:
    p = paths_for_fasta(fasta)
    if exists_nonempty(p["tomtom_tsv"]):
        df = pd.read_csv(p["tomtom_tsv"], sep="\t", comment="#")
        df["dataset"] = p["name"]
        tomtom_tables.append(df)

tomtom_all = pd.concat(tomtom_tables, ignore_index=True) if tomtom_tables else pd.DataFrame()
tomtom_all.head()

In [ ]:
tomtom_all.sort_values(["dataset", "q-value"]).head(30)

### FIMO-JASPAR

In [ ]:
fimo_jaspar_tables = []

for fasta in fastas:
    p = paths_for_fasta(fasta)
    if exists_nonempty(p["fimo_jaspar_tsv"]):
        df = pd.read_csv(p["fimo_jaspar_tsv"], sep="\t", comment="#")
        df["dataset"] = p["name"]
        fimo_jaspar_tables.append(df)

fimo_jaspar_all = pd.concat(fimo_jaspar_tables, ignore_index=True) if fimo_jaspar_tables else pd.DataFrame()
fimo_jaspar_all.head()

In [ ]:
if not fimo_jaspar_all.empty:
    fimo_jaspar_all.groupby("dataset").size().sort_values(ascending=False)

### FIMO-STREME

In [ ]:
fimo_streme_tables = []

for fasta in fastas:
    p = paths_for_fasta(fasta)
    if exists_nonempty(p["fimo_streme_tsv"]):
        df = pd.read_csv(p["fimo_streme_tsv"], sep="\t", comment="#")
        df["dataset"] = p["name"]
        fimo_streme_tables.append(df)

fimo_streme_all = pd.concat(fimo_streme_tables, ignore_index=True) if fimo_streme_tables else pd.DataFrame()
fimo_streme_all.head()

## Save Results

In [ ]:
SUMMARY_DIR = RESULT_DIR / "summary_tables"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

status.to_csv(SUMMARY_DIR / "meme_pipeline_status.csv", index=False)

if not tomtom_all.empty:
    tomtom_all.to_csv(SUMMARY_DIR / "tomtom_all.tsv", sep="\t", index=False)

if not fimo_jaspar_all.empty:
    fimo_jaspar_all.to_csv(SUMMARY_DIR / "fimo_jaspar_all.tsv", sep="\t", index=False)

if not fimo_streme_all.empty:
    fimo_streme_all.to_csv(SUMMARY_DIR / "fimo_streme_all.tsv", sep="\t", index=False)

print("Saved summary tables to:", SUMMARY_DIR)